# Bokmål/nynorsk alignment av lånekassens dokumenter med LaBSE

In [ ]:
import pandas as pd

df = pd.read_json("lånekassen_data.json")
df

In [ ]:
nynorske = df[df.lang == "nno"].copy()
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"].copy()
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

Lim sammen avsnittene i hvert dokument

In [ ]:
nynorske["texts_joined"] = nynorske.fulltext.apply(lambda x: "\n".join(x))
bokmålske["texts_joined"] = bokmålske.fulltext.apply(lambda x: "\n".join(x))

Last inn fasit

In [ ]:
hash_to_i_nn = {e.doc_hash: e.Index for e in nynorske.itertuples()}
hash_to_i_bm = {e.doc_hash: e.Index for e in bokmålske.itertuples()}

fasit = pd.read_csv("lanekassen_fasit.csv")
fasit_set = {(hash_to_i_nn[nn_doc_hash], hash_to_i_bm[bm_doc_hash]) for nn_doc_hash, bm_doc_hash in zip(fasit.nynorsk_doc_hash, fasit.bokmål_doc_hash)}

def compare_matches_to_fasit(matches):
    matches = {(i, match["corpus_id"]) for i, match in matches}

    hits = fasit_set.intersection(matches)
    misses = fasit_set - matches
    
    return (hits, misses)

Last inn modellen

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('sentence-transformers/LaBSE', device="cuda")

# Tell hvor mange dokumenter og avsnitt som er for lange for modellen

In [ ]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

nynorsk_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in nynorske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in nynorsk_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(nynorsk_documents_token_sequence_lengths)} nynorske dokumentene er:
    {under_max_len} under LaBSE sin makslengde 
    {over_max_len}  over LaBSE sin makslengde 
Altså er {round(under_max_len/len(nynorsk_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

bokmål_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in bokmålske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in bokmål_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(bokmål_documents_token_sequence_lengths)} dokumentene på bokmål er:
    {under_max_len} under LaBSE sin makslengde 
    {over_max_len}  over LaBSE sin makslengde 
Altså er {round(under_max_len/len(bokmål_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

In [ ]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

token_sequence_lengths = df.fulltext.apply(lambda x: [len(tokenizer.tokenize(e)) for e in x])

under_max_len = 0
over_max_len = 0
zero_len = 0
for e in token_sequence_lengths:
    for token_len in e:
        if token_len:
            if token_len > max_len:
                over_max_len += 1
            else:
                under_max_len += 1
        else:
            zero_len += 1

print(f"""
Det er totalt {sum((under_max_len, over_max_len))} ikke-tomme avsnitt/setninger (og {zero_len} er tomme)
Av de ikke-tomme avsnittene er:
    {under_max_len} under LaBSE sin makslengde 
    {over_max_len}  over LaBSE sin makslengde 
Altså er {round(under_max_len/sum((under_max_len, over_max_len))*100, 2)}% av de ikke-tomme avsnittene under makslengden 
""")


# Dokumentalignment
Finn den likeste bokmål-dokument-embeddingen for hver nynorsk-dokument-embedding.  

In [ ]:
from pathlib import Path 

def write_doc_matches_to_file(matches, filename):
    file_path = Path(filename)
    file_path.parent.mkdir(exist_ok=True, parents=True)
    
    nn_i, bm_i = zip(*[(i, res["corpus_id"]) for i, res in matches])
                    
    nn_hashes = list(nynorske.doc_hash.iloc[list(nn_i)])
    bm_hashes = list(bokmålske.doc_hash.iloc[list(bm_i)])

    pd.DataFrame({"nynorsk_doc_hash": nn_hashes, "bokmål_doc_hash": bm_hashes}).to_csv(filename, index=False)

In [ ]:
doc_alignment_results = {}

## Aksepter cut-off

Send dokumentet as is til modellen (vil kuttes av på modellens makslengde)

In [ ]:
import numpy as np 
from pathlib import Path

base_path = "labse/texts_joined"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    bokmål_embeddings = model.encode(bokmålske.texts_joined)
    nynorsk_embeddings = model.encode(nynorske.texts_joined)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")


hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["naiv_cutoff"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser resultater 

In [ ]:
from utils import print_matches, print_misses

# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

#### Falsk positiv
Eksempel på en falsk positiv.  
Noen lister av datoer og banker og renter blir veldig like  

In [ ]:
i = 21
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

#### Falsk negativ

In [ ]:
i = 80
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

## Del opp dokumentene i biter mindre enn modellens makslengde og aggreger

In [ ]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in nynorske.texts_joined]
bokmål_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in bokmålske.texts_joined]

In [ ]:
# from collections import Counter 
# pd.DataFrame(Counter([len(e) for e in nynorsk_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")
# pd.DataFrame(Counter([len(e) for e in bokmål_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")

### Mean pooling

In [ ]:
import numpy as np 

base_path = "labse/maxlen_parts_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["mean_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

#### Inspiser resultater

In [ ]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

In [ ]:
for nn_i, bm_i in misses:
    match_i = search_result[nn_i][0]["corpus_id"]
    match_score = search_result[nn_i][0]["score"]
    
    if match_i != bm_i:
        print("Likeste søketreff er et annet dokument enn fasit\n")
        print(f"Fasit indeks: {bm_i}\nTreff indeks: {match_i}")
        print(f"Søketekst og fasit likhet: {float(util.cos_sim(nynorsk_embeddings[nn_i], bokmål_embeddings[bm_i]))}")
        print(f"Søketekst og match likhet: {match_score}")
        print(f"Treff og fasit likhet {float(util.cos_sim(bokmål_embeddings[bm_i], bokmål_embeddings[match_i]))}\n\n")

        print(f"Nynorsk søketekst:\n\t{nynorske.texts_joined[nn_i][:300]}\n_____")
        print(f"Bokmål søketreff:\n\t{bokmålske.texts_joined[match_i][:300]}\n_____")
        print(f"Bokmål fasit:\n\t{bokmålske.texts_joined[bm_i][:300]}\n_____")

    else:
        print("Likeste søketreff er det samme som fasiten, men similarity score var under terskelen\n")
        assert match_score <= threshold

##### Falsk positiv:

In [ ]:
i = 19
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

##### Falsk negativ

In [ ]:
i = 72
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

### Max pooling

In [ ]:
import numpy as np 

base_path = "labse/maxlen_parts_max_pooling"
emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")


hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["max_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

#### Inspiser resultater

In [ ]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

##### Falsk positiv

In [ ]:
i = 19
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

##### Falske negativer
Veldig lav score!

In [ ]:
i = 72
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

In [ ]:
i = 80
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

## Konklusjon

Vi får like mange treff på fasit med naiv cutoff og mean pooling.  
Men naiv cutoff treffer litt flere av den totale mengden dokumenter.  
Vi har ikke gulldata å sammenlikne med, så det er ikke så godt å si helt sikkert om dette er den beste metoden.

In [ ]:
pd.DataFrame(doc_alignment_results).T.sort_values("hits")

# Setnings/avsnittsalignment

In [ ]:
sent_alignment_results = {}

In [ ]:
from collections import defaultdict

nynorske_sentences = defaultdict(list)
for t, df_ in nynorske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        nynorske_sentences["text"].append(t)
        nynorske_sentences["doc_hashes"].append(set(df_.doc_hash))
        nynorske_sentences["urls"].append(set(df_.url))

nynorske_flat = pd.DataFrame(nynorske_sentences)

bokmålske_sentences = defaultdict(list)
for t, df_ in bokmålske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        bokmålske_sentences["text"].append(t)
        bokmålske_sentences["doc_hashes"].append(set(df_.doc_hash))
        bokmålske_sentences["urls"].append(set(df_.url))

bokmålske_flat = pd.DataFrame(bokmålske_sentences)

In [ ]:
len(bokmålske_flat), len(nynorske_flat)

In [ ]:
def write_sent_matches_to_file(matches, filename):
    nn_i, bm_i = zip(*[(i, res["corpus_id"]) for i, res in matches])

    nn = nynorske_flat.iloc[list(nn_i)].rename(mapper=lambda x: "nn_"+x, axis=1)
    bm = bokmålske_flat.iloc[list(bm_i)].rename(mapper=lambda x: "bm_"+x, axis=1)
    nn.index = range(len(nn))
    bm.index = range(len(bm))
    
    pd.concat([nn, bm], axis=1).to_csv(filename, index=False)


In [ ]:
same_text = bokmålske_flat.merge(nynorske_flat, on="text", suffixes=["_bm", "_nn"])
sent_alignment_results["string_comparison"] = {"matches": len(same_text), "percent of sents": round(len(same_text)/len(nynorske_flat), 2)}

## Aksepter cut-off
Godta cut-off på LABSE sin maxlengde

In [ ]:
import numpy as np

base_path = "labse/texts_flat"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = model.encode(nynorske_flat.text)
    bokmål_embeddings = model.encode(bokmålske_flat.text)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["naiv_cutoff"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorske_flat.text)*100, 2)}

### Inspiser resultater

In [ ]:
from utils import print_matches
 
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)

## Del opp setninger/avsnitt som er lengre enn LABSE sin maxlengde og agregger

In [ ]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in nynorske_flat.text]
bokmål_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in bokmålske_flat.text]

### Mean pooling

In [ ]:
base_path = "/labse/maxlen_parts_flat_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["mean_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

#### Inspiser resultater

In [ ]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)


### Max pooling

In [ ]:
base_path = "labse/maxlen_parts_flat_max_pooling"
emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["max_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

#### Inspiser resultater

In [ ]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)


## Konklusjon
Vi får flest matches med naiv cutoff (vi vet at over 93% av avsnittene er innenfor modellens makslengde), etterfulgt av mean pooling.  
Vi har ikke gulldata å sammenlikne med, så det er ikke så godt å si helt sikkert om dette er den beste metoden.

In [ ]:
pd.DataFrame(sent_alignment_results).T.sort_values("matches")